[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pmarcelino/mobillity-courses/blob/main/mobillity-univ/module-3-cleaning-and-exploring/notebook-3.5-trips.ipynb)


# Module 3 — Exploring Trip Data with an LLM (Colab walkthrough)

A hands-on companion to the lecture on **exploring data with an LLM**, run on a small, **synthetic** bike-share **trips** table (already clean and joined).

## What this notebook is for

This notebook is a **preview** of what the lecture will show you. Run it top to bottom — or just read the printed outputs — to get a feel for the work before the lecture explains it: turning a vague curiosity into one specific question at a time, and letting each answer suggest the next. You don't need to follow every line of code yet. The goal is to **picture what we'll be doing**, so when the lecture walks through the *why*, you already have a concrete example in your head. Every step prints a result you can check.

### How to run in Google Colab
1. `Runtime ▸ Run all`. The first code cell **generates the CSV** (seeded, so results are identical every run) into the current folder — no upload needed.
2. Everything below reads that CSV and prints a checkable result for each thing the lecture shows.

> ⚠️ **The data is synthetic.** Rows are fabricated to be *plausible* and to match the columns, types, quirks, and numbers used in the Module 3 lecture scripts. It is **not** real Bicing data — real Bicing snapshots are ~370 MB (too big for Colab) and no public trip-level Bicing data exists, which is why this data is generated below.


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")            # headless backend (works in Colab and CI)
import matplotlib.pyplot as plt

# Where the CSV is written/read. Default "." works in Colab (/content).
DATA_DIR = os.environ.get("MODULE3_DATA_DIR", ".")
os.makedirs(DATA_DIR, exist_ok=True)
print("DATA_DIR =", DATA_DIR)


DATA_DIR = /private/tmp/claude-501/-Users-pedromarcelino-Documents-xval-bmad-ed-courses-mobillity-univ/3635fdd0-7a4e-4381-b73d-6946be2a9b5e/scratchpad/exec_dir


## Section 0 — Generate the dataset

This cell creates the CSV used below. The generator is seeded
(`np.random.default_rng(42)`), so every run produces the exact same data.


In [2]:
import numpy as np
import pandas as pd

SEED = 42

BCN_STREETS = [
    "Gran Via Corts Catalanes", "Roger de Flor", "Napols", "Carrer de Arago",
    "Passeig de Gracia", "Carrer de Balmes", "Rambla Catalunya", "Carrer de Muntaner",
    "Avinguda Diagonal", "Carrer de Sardenya", "Carrer de Provenca", "Carrer de Valencia",
    "Carrer de Mallorca", "Passeig de Sant Joan", "Carrer de Pau Claris", "Via Laietana",
    "Carrer de Tarragona", "Carrer de Sants", "Avinguda Meridiana", "Carrer del Rossello",
]

def build_trips(rng):
    """Clean, joined bike-share trips (synthetic) — story 3.5 EDA."""
    n = 5000
    dates = pd.Timestamp("2019-03-01") + pd.to_timedelta(rng.integers(0, 31, n), unit="D")
    weekend = dates.dayofweek.to_numpy() >= 5

    wk_h = np.array([6, 7, 8, 9, 12, 14, 17, 18, 19, 21])
    wk_p = np.array([2, 5, 11, 6, 4, 3, 6, 11, 6, 2], float); wk_p /= wk_p.sum()
    we_h = np.array([9, 10, 11, 12, 13, 14, 15, 16, 17, 19])
    we_p = np.array([3, 5, 7, 9, 9, 8, 7, 6, 4, 3], float); we_p /= we_p.sum()
    hours = np.where(weekend, rng.choice(we_h, n, p=we_p), rng.choice(wk_h, n, p=wk_p))
    start_time = dates + pd.to_timedelta(hours * 60 + rng.integers(0, 60, n), unit="m")

    st_p = np.linspace(3, 1, len(BCN_STREETS)); st_p /= st_p.sum()
    start_station = rng.choice(BCN_STREETS, n, p=st_p)
    end_station = rng.choice(BCN_STREETS, n)

    dur = np.where(weekend, rng.normal(24, 8, n), rng.normal(12, 4, n))
    dur = np.clip(np.round(dur), 2, None).astype(int)

    order = np.argsort(start_time.values)
    df = pd.DataFrame({
        "trip_id": "", "start_time": start_time.strftime("%Y-%m-%d %H:%M:%S"),
        "start_station": start_station, "end_station": end_station, "duration_min": dur,
    }).iloc[order].reset_index(drop=True)
    df["trip_id"] = [f"T{100000 + i}" for i in range(n)]
    return df

rng = np.random.default_rng(SEED)
trips_raw = build_trips(rng)
trips_raw.to_csv(f"{DATA_DIR}/trips-sample.csv", index=False)
print("Generated trips-sample.csv :", len(trips_raw), "rows  (story 3.5)")

Generated trips-sample.csv : 5000 rows  (story 3.5)


## Exploring Data with LLMs

The data is already clean and joined, so we stop asking *what is this data?* and start
asking *what does it tell us?* — one specific question at a time, each answer shaping the
next.


In [3]:
trips = pd.read_csv(f"{DATA_DIR}/trips-sample.csv", parse_dates=["start_time"])
trips["hour_of_day"] = trips["start_time"].dt.hour

# Q1: when are peak usage hours?
by_hour = trips.groupby("hour_of_day").size()
print("trips by hour (top 5):")
print(by_hour.sort_values(ascending=False).head(5).to_string())
print("busiest hour:", int(by_hour.idxmax()))

fig, ax = plt.subplots(figsize=(8, 3))
by_hour.plot(kind="bar", ax=ax); ax.set_title("Trips by hour of day"); ax.set_xlabel("hour")
fig.tight_layout(); fig.savefig(f"{DATA_DIR}/fig_3.5_trips_by_hour.png", dpi=90); plt.close(fig)
print("saved:", "fig_3.5_trips_by_hour.png")


trips by hour (top 5):
hour_of_day
18    659
8     646
12    480
17    465
19    451
busiest hour: 18
saved: fig_3.5_trips_by_hour.png


In [4]:
# Q2: which stations dominate at peak hours?
peak_hours = by_hour.sort_values(ascending=False).head(4).index.tolist()
peak = trips[trips["hour_of_day"].isin(peak_hours)]
top_stations = peak.groupby("start_station").size().sort_values(ascending=False).head(5)
print("peak window hours:", sorted(peak_hours))
print("top stations at peak:")
print(top_stations.to_string())


peak window hours: [8, 12, 17, 18]
top stations at peak:
start_station
Gran Via Corts Catalanes    188
Carrer de Arago             157
Roger de Flor               156
Napols                      151
Passeig de Gracia           149


In [5]:
# Q3: is trip duration different on weekends vs weekdays?
trips["is_weekend"] = trips["start_time"].dt.dayofweek >= 5
summary = trips.groupby("is_weekend")["duration_min"].agg(["mean", "median", "std"])
summary.index = ["weekday", "weekend"]
print(summary.round(1).to_string())
wk = trips.loc[~trips["is_weekend"], "duration_min"].mean()
we = trips.loc[trips["is_weekend"], "duration_min"].mean()
print(f"mean duration: weekday {wk:.1f} min vs weekend {we:.1f} min -> weekend trips are longer")

fig, ax = plt.subplots(figsize=(6, 3.2))
trips.boxplot(column="duration_min", by="is_weekend", ax=ax)
ax.set_title("Trip duration: weekday (False) vs weekend (True)"); plt.suptitle("")
fig.tight_layout(); fig.savefig(f"{DATA_DIR}/fig_3.5_weekend_duration.png", dpi=90); plt.close(fig)
print("saved:", "fig_3.5_weekend_duration.png")


         mean  median  std
weekday  11.9    12.0  4.0
weekend  24.2    24.0  8.1
mean duration: weekday 11.9 min vs weekend 24.2 min -> weekend trips are longer
saved: fig_3.5_weekend_duration.png


## Done — trip exploration

The exploration lecture has run on the synthetic trips: trips-by-hour with the commute
peak, the busiest stations inside the peak window, and the weekend-vs-weekday duration
difference. Each answer is the kind of result that shapes the next question. Re-run any
time; the seeded generator reproduces identical numbers. Then try the matching exercise
yourself.
